In [1]:
import json
import operator
from typing import Annotated, Literal, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


# ==========================================
# 1. State 定义
# ==========================================
class ApprovalState(TypedDict):
    request_id: str
    employee_name: str
    amount: float
    manager_approved: bool | None
    status: str
    logs: Annotated[list[str], operator.add]


# ==========================================
# 2. Node 定义
# ==========================================
def prepare_request(state: ApprovalState) -> dict:
    return {
        "status": "waiting_manager_approval",
        "logs": [f"申请 {state['request_id']} 已提交经理审批"],
    }


def manager_approval(state: ApprovalState) -> dict:
    # 挂起流程，等待外部传入决策
    # 注意：恢复执行时本节点会从头重新执行，因此 interrupt 之前不要有任何非幂等的副作用
    decision = interrupt({
        "request_id": state["request_id"],
        "employee_name": state["employee_name"],
        "amount": state["amount"],
        "question": "是否批准该申请？",
    })

    approved = bool(decision.get("approved")) if isinstance(decision, dict) else bool(decision)

    return {
        "manager_approved": approved,
        "status": "manager_approved" if approved else "manager_rejected",
        "logs": ["经理审批通过" if approved else "经理审批拒绝"],
    }


def hr_process(state: ApprovalState) -> dict:
    return {
        "status": "completed",
        "logs": ["HR 已完成后续处理"],
    }


def rejected(state: ApprovalState) -> dict:
    return {
        "status": "rejected",
        "logs": ["流程结束：经理拒绝"],
    }


# ==========================================
# 3. Router 定义
# ==========================================
def route_after_approval(
    state: ApprovalState,
) -> Literal["hr_process", "rejected"]:
    if state["manager_approved"]:
        return "hr_process"
    return "rejected"


# ==========================================
# 4. 构建 Graph
# ==========================================
def build_graph():
    builder = StateGraph(ApprovalState)

    builder.add_node("prepare_request", prepare_request)
    builder.add_node("manager_approval", manager_approval)
    builder.add_node("hr_process", hr_process)
    builder.add_node("rejected", rejected)

    builder.add_edge(START, "prepare_request")
    builder.add_edge("prepare_request", "manager_approval")

    builder.add_conditional_edges(
        "manager_approval",
        route_after_approval,
        {
            "hr_process": "hr_process",
            "rejected": "rejected",
        },
    )

    builder.add_edge("hr_process", END)
    builder.add_edge("rejected", END)

    return builder.compile(checkpointer=InMemorySaver())


# ==========================================
# 5. 测试辅助函数
# ==========================================
def print_snapshot_info(stage: str, snapshot):
    print(f"\n--- 快照检查 [{stage}] ---")
    print(f"snapshot.values: {json.dumps(snapshot.values, ensure_ascii=False)}")
    print(f"snapshot.next: {snapshot.next}")
    
    # 获取中断信息 (兼容不同版本的 interrupts 属性访问模式)
    interrupts_data = getattr(snapshot, "interrupts", None)
    if not interrupts_data and snapshot.tasks:
        interrupts_data = [task.interrupts for task in snapshot.tasks if task.interrupts]
    print(f"snapshot.interrupts: {interrupts_data}")
    print("-" * 30)


# ==========================================
# 6. 测试用例定义
# ==========================================
def test_1_approved(graph):
    print("=" * 65)
    print("▶ 测试 1：经理批准 → hr_process → completed")
    print("=" * 65)

    config = {"configurable": {"thread_id": "approval-thread-001"}}
    initial_state: ApprovalState = {
        "request_id": "REQ-001",
        "employee_name": "Kevin",
        "amount": 50000.0,
        "manager_approved": None,
        "status": "created",
        "logs": [],
    }

    # 1. 运行至暂停点
    print("\n[阶段 1] 启动工作流...")
    graph.invoke(initial_state, config=config)

    # 打印暂停快照
    snap_before = graph.get_state(config)
    print_snapshot_info("暂停等待人工输入", snap_before)

    # 2. 传入批准决策恢复流程
    print("\n[阶段 2] 模拟经理批准，恢复执行...")
    result = graph.invoke(
        Command(resume={"approved": True, "approver": "李经理"}),
        config=config,
    )

    # 打印最终快照
    snap_after = graph.get_state(config)
    print_snapshot_info("流程终结", snap_after)

    print("\n最终 State Logs:")
    for log in result["logs"]:
        print(" -", log)
    print("\n")


def test_2_rejected(graph):
    print("=" * 65)
    print("▶ 测试 2：经理拒绝 → rejected → END")
    print("=" * 65)

    config = {"configurable": {"thread_id": "approval-thread-002"}}
    initial_state: ApprovalState = {
        "request_id": "REQ-002",
        "employee_name": "Kevin",
        "amount": 120000.0,
        "manager_approved": None,
        "status": "created",
        "logs": [],
    }

    # 1. 运行至暂停点
    print("\n[阶段 1] 启动工作流...")
    graph.invoke(initial_state, config=config)

    # 打印暂停快照
    snap_before = graph.get_state(config)
    print_snapshot_info("暂停等待人工输入", snap_before)

    # 2. 传入拒绝决策恢复流程
    print("\n[阶段 2] 模拟经理拒绝，恢复执行...")
    result = graph.invoke(
        Command(resume={"approved": False, "approver": "张经理"}),
        config=config,
    )

    # 打印最终快照
    snap_after = graph.get_state(config)
    print_snapshot_info("流程终结", snap_after)

    print("\n最终 State Logs:")
    for log in result["logs"]:
        print(" -", log)
    print("\n")


if __name__ == "__main__":
    app = build_graph()
    test_1_approved(app)
    test_2_rejected(app)

▶ 测试 1：经理批准 → hr_process → completed

[阶段 1] 启动工作流...

--- 快照检查 [暂停等待人工输入] ---
snapshot.values: {"request_id": "REQ-001", "employee_name": "Kevin", "amount": 50000.0, "manager_approved": null, "status": "waiting_manager_approval", "logs": ["申请 REQ-001 已提交经理审批"]}
snapshot.next: ('manager_approval',)
snapshot.interrupts: (Interrupt(value={'request_id': 'REQ-001', 'employee_name': 'Kevin', 'amount': 50000.0, 'question': '是否批准该申请？'}, id='07e989a3b8af091156cd0852ed3c6d08'),)
------------------------------

[阶段 2] 模拟经理批准，恢复执行...

--- 快照检查 [流程终结] ---
snapshot.values: {"request_id": "REQ-001", "employee_name": "Kevin", "amount": 50000.0, "manager_approved": true, "status": "completed", "logs": ["申请 REQ-001 已提交经理审批", "经理审批通过", "HR 已完成后续处理"]}
snapshot.next: ()
snapshot.interrupts: ()
------------------------------

最终 State Logs:
 - 申请 REQ-001 已提交经理审批
 - 经理审批通过
 - HR 已完成后续处理


▶ 测试 2：经理拒绝 → rejected → END

[阶段 1] 启动工作流...

--- 快照检查 [暂停等待人工输入] ---
snapshot.values: {"request_id": "REQ-002", "emplo